# AMP Quickstart — try it on a real NVIDIA GPU, zero setup

Clones AMP straight from GitHub, builds the CUDA backend, runs the FP32 GEMM correctness check, validates the one-command kernel-check flow (`amp_check.sh`), and produces `cuda_dump.json` for a cross-vendor parity check against an AMD-side dump.

**Before running anything else:** Runtime -> Change runtime type -> T4 GPU (free tier is fine). If you skip this, the next cell fails with `nvidia-smi: command not found` — that just means the runtime is still CPU-only.

**Then: Runtime -> Run all.** No file upload, no local setup.

In [ ]:
import shutil, sys
if shutil.which('nvidia-smi') is None:
    sys.exit("No GPU detected. Go to Runtime -> Change runtime type -> select a GPU (T4), "
             "then Runtime -> Restart session, then re-run from this cell.")
!nvidia-smi
!nvcc --version

In [ ]:
# AMP source: public repo, cloned directly -- no manual upload needed
!rm -rf /content/amp
!git clone --depth 1 https://github.com/agusbass/AMP.git /content/amp
!ls /content/amp

In [ ]:
# CMake on Colab's default image is often older than the 3.20 this
# project requires (CMakeLists.txt:1) -- upgrade unconditionally via pip,
# it's the fastest path and doesn't need apt/sudo.
!pip install -q --upgrade cmake
!hash -r
!cmake --version

In [ ]:
# Detect GPU compute capability to pick the right -arch flag
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip().splitlines()[0]
cc = out.replace('.', '')
print('Compute capability:', out, '-> sm_' + cc)

In [ ]:
%%bash -s "$cc"
cd /content/amp
rm -rf build && mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DAMP_BACKEND=CUDA -DCMAKE_CUDA_ARCHITECTURES=$1 -DAMP_HAVE_CUBLAS=ON 2>&1 | tail -30
cmake --build . -j"$(nproc)" --target amp_verify_matmul amp_parity_dump test_triple amp_validate_kernel 2>&1 | tail -150

If the build above fails, paste the error output back and we'll fix itin the repo, regenerate `amp_cuda_test.zip`, and you re-upload + re-runfrom Cell 3. Otherwise, run the checks below.

In [ ]:
!/content/amp/build/test_triple

## Numerical correctness check (GPU vs CPU reference)Proves the FP32 GEMM tile fix is numerically correct on this vendor, inisolation (same check that passed on real AMD MI300X withmax_rel_err≈0.00006 -- this run shows whether NVIDIA matches that).

In [ ]:
!/content/amp/build/amp_verify_matmul

## Cross-vendor parity dumpDumps the actual GEMM output + GFLOPS to JSON, so it can be diffeddirectly against the AMD/HIP dump (not just each vendor's ownCPU-reference check).

In [ ]:
# 96^3 dump (matches the original cuda_dump.json already collected)
!/content/amp/build/amp_parity_dump /content/amp/cuda_dump.json

# 1024^3 dump -- more representative of real LLM-inference matmul sizes
!/content/amp/build/amp_parity_dump /content/amp/cuda_dump_1024.json 1024 1024 1024
!cat /content/amp/cuda_dump_1024.json | head -c 500

In [ ]:
from google.colab import files
files.download('/content/amp/cuda_dump.json')
files.download('/content/amp/cuda_dump_1024.json')

## Next stepWith `cuda_dump.json` from this notebook and `hip_dump.json` from theMI300X run, diff them locally:```bashpython3 scripts/parity_check.py cuda_dump.json hip_dump.json --analyze```

## NEW: validate YOUR OWN kernel (not just AMP's bundled example)

`amp_validate_kernel` dlopen's a shared library you build yourself (your own GEMM kernel, CUDA or HIP) and checks it against a CPU reference -- this is the workflow a real user follows, as opposed to the cells above which only exercise AMP's own reference kernel.

`examples/user_plugin_example.cu` is a complete, runnable wrapper showing the exact shape required.

In [ ]:
%%bash
cd /content/amp
nvcc -shared -Xcompiler -fPIC examples/user_plugin_example.cu -o build/libexample_cuda.so 2>&1
echo BUILD_OK
ls -la build/libexample_cuda.so

In [ ]:
!/content/amp/build/amp_validate_kernel /content/amp/build/libexample_cuda.so /content/amp/user_cuda_dump.json 512 512 512

If this prints PASS with a small max_rel_err, the plugin interface works end to end on real NVIDIA hardware. The same `libexample_*.so` shape built with `hipcc` instead of `nvcc` on an AMD machine produces a dump comparable via `scripts/parity_check.py` -- this is the path a real user takes with their own kernel instead of `examples/user_plugin_example.cu`.

## NEW: one-command kernel check (amp_check.sh)

Tests the new one-shot UX: write a kernel matching the common GEMM signature, run ONE command, get a validated dump -- no manual wrapper editing, no separate build step typed out by hand.

In [ ]:
%%writefile /content/amp/my_kernel.cu
__global__ void my_naive_gemm(const float* A, const float* B, float* C,
                               int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= M || col >= N) return;
    float acc = 0.0f;
    for (int k = 0; k < K; ++k) acc += A[row * K + k] * B[k * N + col];
    C[row * N + col] = acc;
}

In [ ]:
!cd /content/amp && bash scripts/amp_check.sh my_kernel.cu my_naive_gemm 512 512 512

If this prints PASS, the one-command flow works end to end: detect toolchain -> wrap -> compile -> build AMP if needed -> validate vs CPU reference -> write cuda_dump.json. This is the exact command a real user runs with their own kernel, on either a CUDA or AMD machine.